# Test: CodingAgent Smoke Test

Verify that `src/coding_agent.py` works end-to-end:
1. Code execution tools (Python, R, Bash)
2. LangGraph agent loop with a simple task
3. Full aging-analysis pipeline on one study

In [ ]:
import os, sys

# Add src/ to path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from coding_agent import (
    run_python_repl,
    run_r_code,
    run_bash_script,
    run_with_timeout,
    CodingAgent,
)

print("Imports OK")
print(f"REPO_ROOT: {REPO_ROOT}")

## 1. Test code execution tools

In [ ]:
# --- Python REPL ---
out = run_python_repl('print(2 + 2)')
assert out.strip() == '4', f"Expected '4', got {out!r}"
print(f"Python REPL: {out.strip()}  ✓")

# Persistent namespace: variable should survive across calls
run_python_repl('x = 42')
out2 = run_python_repl('print(x)')
assert out2.strip() == '42', f"Expected '42', got {out2!r}"
print(f"Persistent namespace: x = {out2.strip()}  ✓")

In [ ]:
# --- Bash ---
out = run_bash_script('echo hello')
assert 'hello' in out, f"Expected 'hello' in output, got {out!r}"
print(f"Bash: {out.strip()}  ✓")

In [ ]:
# --- R (optional, may not be installed) ---
out = run_r_code('cat(1 + 1)')
if 'Error' in out:
    print(f"R not available: {out.strip()}  (skipped)")
else:
    assert '2' in out, f"Expected '2', got {out!r}"
    print(f"R: {out.strip()}  ✓")

In [ ]:
# --- Timeout ---
out = run_with_timeout(run_python_repl, ['print("fast")'], timeout=5)
assert 'fast' in out
print(f"Timeout (fast): {out.strip()}  ✓")

out = run_with_timeout(run_python_repl, ['import time; time.sleep(10); print("done")'], timeout=2)
assert 'timed out' in out.lower() or 'TIMEOUT' in out
print(f"Timeout (slow): correctly timed out  ✓")

## 2. Test agent loop with a simple task

Give the agent a trivial math problem to verify the generate → execute → observe loop works.

In [ ]:
LLM_MODEL = "claude-sonnet-4-20250514"

agent = CodingAgent(llm=LLM_MODEL, timeout=60)
agent.configure()
print(f"Agent configured (model={LLM_MODEL})")

In [ ]:
%%time
log, final = agent.go(
    "Compute the sum of the first 100 positive integers using Python. "
    "Return the answer as <solution>NUMBER</solution>."
)

print(f"\n--- Final message (last 300 chars) ---")
print(final[-300:])
print(f"\nAgent took {len(log)} turns")

## 3. Full pipeline test: analyze one GSE study

Run the agent on a real aging-analysis prompt for a single study.

In [ ]:
import pandas as pd

BASE_DIR = REPO_ROOT
DATA_DIR = os.path.join(BASE_DIR, "data")

# Verify data files
for p in [
    os.path.join(DATA_DIR, "dataset_level", "gse_rna_mm.csv"),
    os.path.join(DATA_DIR, "clock_output", "mouse_rna_age_short.csv"),
]:
    assert os.path.isfile(p), f"Missing: {p}"
    print(f"  [OK] {p}")

In [ ]:
gse_df = pd.read_csv(os.path.join(DATA_DIR, "dataset_level", "gse_rna_mm.csv"))
study_id = "GSE122080"
study_info = gse_df[gse_df["accession"] == study_id]
print(f"Study: {study_id}")
print(study_info[["accession", "title"]].to_string(index=False))

In [ ]:
output_dir = os.path.join(BASE_DIR, "test_results", study_id)
os.makedirs(output_dir, exist_ok=True)

prompt = f"""Analyze the aging effects of conditions from study {study_id}.

Workflow:
1. Load dataset overview from: {DATA_DIR}/dataset_level/gse_rna_mm.csv (filter by accession == '{study_id}')
2. Extract metadata from: {DATA_DIR}/GEO_mm_rna/processed_metadata_preharmonize/{study_id}.csv
3. Import clock data from: {DATA_DIR}/clock_output/mouse_rna_age_short.csv
4. Merge metadata with clock data on GSM ID.
5. For each condition vs control comparison, run a t-test on these clocks:
   - Hazard_Rodents_All_All_scaleddiff_EN_tAge
   - Hazard_Mouse_All_All_scaleddiff_EN_tAge
6. Save results to: {output_dir}/aging_analysis.csv
   Columns: gse_id, comparison_name, condition_name, clock, p_value, effect_size, direction

Return ONLY a short confirmation when done.
"""
print("Prompt ready. Running agent...")
print(f"Output: {output_dir}/aging_analysis.csv")

In [ ]:
%%time
agent2 = CodingAgent(llm=LLM_MODEL, timeout=300)
agent2.configure()
log2, final2 = agent2.go(prompt)

print(f"\nAgent took {len(log2)} turns")
print(f"Final message (last 300 chars):\n{final2[-300:]}")

In [ ]:
csv_path = os.path.join(output_dir, "aging_analysis.csv")
if os.path.isfile(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Output CSV: {len(df)} rows, columns: {df.columns.tolist()}")
    display(df)
else:
    print(f"No output CSV at {csv_path}")
    if os.path.isdir(output_dir):
        print(f"Files in output dir: {os.listdir(output_dir)}")